Mount Google Drive để đọc PDF folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cài thư viện

In [ ]:
!pip -q install pymupdf rank-bm25 sentence-transformers faiss-cpu hdbscan umap-learn rispy bibtexparser pandas tqdm

Import & cấu hình đường dẫn dữ liệu

In [ ]:
import os, re, json, math, hashlib
from pathlib import Path
import pandas as pd
from tqdm import tqdm

DATA_DIR = "/content/drive/MyDrive/papers"  # <-- sửa
assert os.path.exists(DATA_DIR), f"Không thấy folder: {DATA_DIR}"

PDF_EXT = {".pdf"}
RIS_EXT = {".ris"}
BIB_EXT = {".bib"}

def sha1_text(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()[:16]

Quét file trong folder (PDF + RIS/BIB)

In [ ]:
data_path = Path(DATA_DIR)
pdf_files = [p for p in data_path.rglob("*") if p.suffix.lower() in PDF_EXT]
ris_files = [p for p in data_path.rglob("*") if p.suffix.lower() in RIS_EXT]
bib_files = [p for p in data_path.rglob("*") if p.suffix.lower() in BIB_EXT]

print("PDF:", len(pdf_files))
print("RIS:", len(ris_files))
print("BIB:", len(bib_files))

Parse RIS/BIB (nếu có) để lấy title/abstract “chuẩn”

In [ ]:
import rispy
import bibtexparser

def parse_ris_file(path: Path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        entries = rispy.load(f)
    rows = []
    for e in entries:
        title = e.get("title") or e.get("primary_title") or ""
        abstract = e.get("abstract") or ""
        year = e.get("year") or ""
        doi = e.get("doi") or ""
        authors = e.get("authors") or []
        keywords = e.get("keywords") or []
        rows.append({
            "source": "RIS",
            "file_path": str(path),
            "title": title.strip(),
            "abstract": abstract.strip(),
            "year": str(year).strip(),
            "doi": doi.strip(),
            "authors": "; ".join(authors) if isinstance(authors, list) else str(authors),
            "keywords": "; ".join(keywords) if isinstance(keywords, list) else str(keywords),
        })
    return rows

def parse_bib_file(path: Path):
    with open(path, "r", encoding="utf-8", errors="ignore") as bibtex_file:
        bib_database = bibtexparser.load(bibtex_file)
    rows = []
    for e in bib_database.entries:
        title = e.get("title", "")
        abstract = e.get("abstract", "")
        year = e.get("year", "")
        doi = e.get("doi", "")
        authors = e.get("author", "")
        keywords = e.get("keywords", "")
        rows.append({
            "source": "BIB",
            "file_path": str(path),
            "title": re.sub(r"[{}]", "", title).strip(),
            "abstract": abstract.strip(),
            "year": str(year).strip(),
            "doi": doi.strip(),
            "authors": authors.strip(),
            "keywords": keywords.strip(),
        })
    return rows

rows_meta = []
for p in ris_files:
    rows_meta.extend(parse_ris_file(p))
for p in bib_files:
    rows_meta.extend(parse_bib_file(p))

df_meta = pd.DataFrame(rows_meta)
print("Metadata records:", len(df_meta))
df_meta.head(3)

Extract Title/Abstract từ PDF (fallback khi không có RIS/BIB)

In [ ]:
import fitz  # PyMuPDF

def clean_text(x: str) -> str:
    x = re.sub(r"\s+", " ", x).strip()
    return x

def extract_first_pages_text(pdf_path: Path, max_pages=2):
    doc = fitz.open(pdf_path)
    texts = []
    for i in range(min(max_pages, doc.page_count)):
        texts.append(doc.load_page(i).get_text("text"))
    doc.close()
    return "\n".join(texts)

def heuristic_title_abstract_from_text(text: str):
    # Title: lấy dòng đầu tiên đủ dài, tránh tiêu đề trang
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    title = ""
    for l in lines[:30]:
        if len(l) >= 12 and len(l) <= 200 and not re.match(r"^\d+$", l):
            title = l
            break

    # Abstract: tìm từ "Abstract" và cắt tới "Introduction"/"Keywords"/"Methods"...
    low = text.lower()
    abstract = ""
    m = re.search(r"\babstract\b", low)
    if m:
        start = m.end()
        tail = text[start:start+5000]  # giới hạn
        stop_patterns = [
            r"\bkeywords?\b", r"\bintroduction\b", r"\bmethods?\b",
            r"\bmaterials?\b", r"\bresults?\b", r"\bbackground\b"
        ]
        stop_idx = None
        for pat in stop_patterns:
            mm = re.search(pat, tail.lower())
            if mm:
                idx = mm.start()
                stop_idx = idx if stop_idx is None else min(stop_idx, idx)
        abstract = tail[:stop_idx].strip() if stop_idx else tail.strip()

    return clean_text(title), clean_text(abstract)

rows_pdf = []
for pdf in tqdm(pdf_files, desc="Extract PDF (title/abstract)"):
    try:
        t = extract_first_pages_text(pdf, max_pages=2)
        title, abstract = heuristic_title_abstract_from_text(t)
        rows_pdf.append({
            "source": "PDF",
            "file_path": str(pdf),
            "title": title,
            "abstract": abstract,
            "year": "",
            "doi": "",
            "authors": "",
            "keywords": "",
        })
    except Exception as e:
        rows_pdf.append({
            "source": "PDF",
            "file_path": str(pdf),
            "title": "",
            "abstract": "",
            "year": "",
            "doi": "",
            "authors": "",
            "keywords": "",
        })

df_pdf = pd.DataFrame(rows_pdf)
print("PDF records:", len(df_pdf))
df_pdf.head(3)

Gộp metadata + PDF, tạo paper_id, chuẩn hoá text

In [ ]:
def norm_key(title, year, doi):
    key = (doi or "").strip().lower()
    if key:
        return "doi:" + key
    return f"t:{(title or '').strip().lower()}|y:{(year or '').strip()}"

df_all = pd.concat([df_meta, df_pdf], ignore_index=True)
df_all["title"] = df_all["title"].fillna("").map(clean_text)
df_all["abstract"] = df_all["abstract"].fillna("").map(clean_text)
df_all["key"] = df_all.apply(lambda r: norm_key(r["title"], r["year"], r["doi"]), axis=1)

# giữ bản "tốt hơn" (abstract dài hơn)
df_all["abs_len"] = df_all["abstract"].map(len)
df_all = df_all.sort_values(["key","abs_len"], ascending=[True, False]).drop_duplicates("key", keep="first")

# tạo paper_id ổn định
df_all["paper_id"] = df_all.apply(lambda r: sha1_text((r["key"] or "") + "|" + (r["title"] or "")), axis=1)
df_all = df_all.reset_index(drop=True)

print("Total unique papers:", len(df_all))
df_all[["paper_id","source","title","abs_len","file_path"]].head(5)

Keyword query + BM25 (lọc candidate nhanh)

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(s: str):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.split()

df_all["text_ta"] = (df_all["title"] + ". " + df_all["abstract"]).str.strip()

corpus_tokens = [tokenize(x) for x in df_all["text_ta"].tolist()]
bm25 = BM25Okapi(corpus_tokens)

# ---- bạn sửa query ở đây ----
query = "machine learning OR deep learning systematic review outcome"
query_tokens = tokenize(query)

scores = bm25.get_scores(query_tokens)
df_all["bm25"] = scores

TOP_K = min(20000, len(df_all))  # nếu dữ liệu nhỏ hơn thì tự co
df_cand = df_all.sort_values("bm25", ascending=False).head(TOP_K).copy()

print("Candidates:", len(df_cand), "| BM25 max:", df_cand["bm25"].max())
df_cand[["paper_id","bm25","title"]].head(5)

Embedding title+abstract

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMB_MODEL = "intfloat/e5-small-v2"  # nhẹ, 384-dim
model = SentenceTransformer(EMB_MODEL, device="cpu")

def build_passage(text):
    # theo khuyến nghị e5: "passage: ..."
    return "passage: " + text

texts = [build_passage(t) for t in df_cand["text_ta"].tolist()]

emb = model.encode(
    texts,
    batch_size=64,              # giảm nếu RAM yếu
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # rất quan trọng cho cosine/IP
)
emb = emb.astype("float32")
print("Embeddings shape:", emb.shape)

FAISS index để tìm similarity nhanh (phục vụ dedup + cluster)

In [ ]:
import faiss

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine ~ inner product do đã normalize
index.add(emb)

print("FAISS ntotal:", index.ntotal)

Dedup (near-duplicate) bằng similarity threshold

In [ ]:
# Union-Find
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
        self.r = [0]*n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a,b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]:
            self.p[ra] = rb
        elif self.r[ra] > self.r[rb]:
            self.p[rb] = ra
        else:
            self.p[rb] = ra
            self.r[ra] += 1

# dedup parameters
KNN = 10
SIM_TH = 0.95   # bạn có thể chỉnh 0.92–0.97

D, I = index.search(emb, KNN)  # D: similarity, I: neighbor indices
dsu = DSU(len(df_cand))

for i in range(len(df_cand)):
    for jpos in range(1, KNN):  # bỏ chính nó ở vị trí 0
        j = int(I[i, jpos])
        sim = float(D[i, jpos])
        if sim >= SIM_TH:
            dsu.union(i, j)

root = [dsu.find(i) for i in range(len(df_cand))]
df_cand["dup_root"] = root

# gán group id dễ nhìn
root_to_gid = {r: idx for idx, r in enumerate(sorted(set(root)))}
df_cand["dup_group_id"] = df_cand["dup_root"].map(root_to_gid)

# chọn canonical: bm25 cao nhất trong group
df_cand["rank_in_group"] = df_cand.groupby("dup_group_id")["bm25"].rank(ascending=False, method="first")
df_cand["is_canonical"] = df_cand["rank_in_group"] == 1

print("Duplicate groups:", df_cand["dup_group_id"].nunique(), "over", len(df_cand), "candidates")
df_cand[["dup_group_id","is_canonical","bm25","title"]].sort_values(["dup_group_id","is_canonical"], ascending=[True, False]).head(10)

Clustering “topic” trên title+abstract (HDBSCAN + UMAP)

In [ ]:
import umap
import hdbscan

# Option: chỉ cluster các canonical để giảm tải
df_for_cluster = df_cand[df_cand["is_canonical"]].copy()
emb_for_cluster = emb[df_for_cluster.index.values]

print("Clustering on canonical:", len(df_for_cluster))

# UMAP giảm chiều để HDBSCAN ổn định hơn
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.0,
    n_components=10,
    metric="cosine",
    random_state=42
)
emb_umap = reducer.fit_transform(emb_for_cluster)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    metric="euclidean",
    cluster_selection_method="eom"
)
labels = clusterer.fit_predict(emb_umap)

df_for_cluster["cluster_id"] = labels  # -1 là noise
print("Clusters (excluding -1):", len(set(labels)) - (1 if -1 in labels else 0))
df_for_cluster[["paper_id","cluster_id","bm25","title"]].head(5)

Gán cluster_id lại cho toàn bộ candidate (theo canonical của dup group)

In [ ]:
# map dup_group_id -> canonical paper_id -> cluster_id
canon_map = df_for_cluster.set_index("paper_id")["cluster_id"].to_dict()

# lấy canonical paper_id cho từng dup_group_id
canon_pid_by_group = df_cand[df_cand["is_canonical"]].set_index("dup_group_id")["paper_id"].to_dict()

def get_cluster_for_row(r):
    canon_pid = canon_pid_by_group.get(r["dup_group_id"])
    return canon_map.get(canon_pid, -1)

df_cand["cluster_id"] = df_cand.apply(get_cluster_for_row, axis=1)

df_cand[["dup_group_id","cluster_id","is_canonical","bm25","title"]].head(10)

Screening (Include/Exclude)

In [ ]:
MUST_HAVE = ["systematic", "review"]
SHOULD_HAVE = ["outcome", "effect"]
MUST_NOT = ["protocol"]

def contains_all(text, terms):
    t = text.lower()
    return all(term.lower() in t for term in terms)

def contains_any(text, terms):
    t = text.lower()
    return any(term.lower() in t for term in terms)

def screening_rule(text):
    if MUST_NOT and contains_any(text, MUST_NOT):
        return "EXCLUDE", "Matched MUST_NOT"
    if MUST_HAVE and not contains_all(text, MUST_HAVE):
        return "EXCLUDE", "Missing MUST_HAVE"
    score = 0
    if SHOULD_HAVE and contains_any(text, SHOULD_HAVE):
        score += 1
    return ("INCLUDE", f"Score={score}") if score >= 0 else ("UNCERTAIN", f"Score={score}")

decisions = []
reasons = []
for t in df_cand["text_ta"].tolist():
    d, r = screening_rule(t)
    decisions.append(d); reasons.append(r)

df_cand["screen_decision"] = decisions
df_cand["screen_reason"] = reasons

df_cand["screen_decision"].value_counts()

Chọn Included set & chuẩn bị sang full-text

In [ ]:
df_inc = df_cand[df_cand["screen_decision"]=="INCLUDE"].copy()

print("Included:", len(df_inc))
df_inc[["paper_id","cluster_id","bm25","title","file_path"]].head(10)

# Tầng Full-text (chạy cho Included)

Extract full-text từ PDF cho Included (có chunking)

In [ ]:
def extract_full_text(pdf_path: str, max_pages=None):
    doc = fitz.open(pdf_path)
    texts = []
    n = doc.page_count if max_pages is None else min(max_pages, doc.page_count)
    for i in range(n):
        texts.append(doc.load_page(i).get_text("text"))
    doc.close()
    return "\n".join(texts)

def chunk_text(text: str, chunk_size=1200, overlap=200):
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+chunk_size])
        i += max(1, chunk_size - overlap)
    return chunks

rows_chunks = []
for _, r in tqdm(df_inc.iterrows(), total=len(df_inc), desc="Full-text chunking"):
    p = r["file_path"]
    if not p.lower().endswith(".pdf") or (not os.path.exists(p)):
        continue
    try:
        ft = extract_full_text(p)
        chs = chunk_text(ft)
        for k, ch in enumerate(chs):
            rows_chunks.append({
                "paper_id": r["paper_id"],
                "cluster_id": r["cluster_id"],
                "chunk_id": k,
                "text": ch
            })
    except Exception as e:
        continue

df_chunks = pd.DataFrame(rows_chunks)
print("Total chunks:", len(df_chunks), "| papers with chunks:", df_chunks["paper_id"].nunique())
df_chunks.head(3)

RAG retrieval trên chunks

In [ ]:
# embed chunks (có thể tốn thời gian nếu full-text nhiều)
chunk_texts = ["passage: " + t for t in df_chunks["text"].tolist()]
chunk_emb = model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

chunk_index = faiss.IndexFlatIP(chunk_emb.shape[1])
chunk_index.add(chunk_emb)
print("Chunk index:", chunk_index.ntotal)

Extraction schema (trích xuất có cấu trúc) – bản “rule-based + evidence spans”

In [ ]:
EXTRACT_FIELDS = {
    "study_design": "randomized controlled trial OR cohort OR cross-sectional OR case-control OR meta-analysis",
    "sample_size": "sample size OR participants OR n=",
    "population": "population OR patients OR subjects",
    "intervention": "intervention OR treatment OR exposure",
    "outcomes": "outcome OR endpoint OR measure",
    "key_results": "results OR findings OR effect OR significant",
    "limitations": "limitation OR limitations",
}

def embed_query(q: str):
    return model.encode(["query: " + q], normalize_embeddings=True).astype("float32")

def retrieve_top_chunks(paper_id: str, q: str, top_k=5):
    # lọc chunks của paper_id
    idxs = df_chunks.index[df_chunks["paper_id"] == paper_id].to_numpy()
    if len(idxs) == 0:
        return []
    # search toàn cục rồi lấy kết quả thuộc paper_id (cách nhanh hơn là build index per paper, nhưng nặng)
    qv = embed_query(q)
    D, I = chunk_index.search(qv, 50)
    hits = []
    for j, sim in zip(I[0], D[0]):
        if j in idxs:
            hits.append((int(j), float(sim)))
        if len(hits) >= top_k:
            break
    out = []
    for j, sim in hits:
        out.append({
            "chunk_row": j,
            "sim": sim,
            "chunk_id": int(df_chunks.loc[j, "chunk_id"]),
            "text": df_chunks.loc[j, "text"][:800]  # cắt ngắn evidence
        })
    return out

evidence_rows = []
for pid in tqdm(df_inc["paper_id"].unique(), desc="Evidence retrieval"):
    row = {"paper_id": pid}
    for field, q in EXTRACT_FIELDS.items():
        spans = retrieve_top_chunks(pid, q, top_k=3)
        row[field] = spans
    evidence_rows.append(row)

df_evidence = pd.DataFrame(evidence_rows)
df_evidence.head(2)


Tạo Evidence Table dạng “mỗi paper một dòng” (flatten đơn giản)

In [ ]:
def best_span_text(spans):
    if not spans:
        return ""
    return spans[0]["text"]

flat_rows = []
for _, r in df_evidence.iterrows():
    pid = r["paper_id"]
    base = {"paper_id": pid}
    for f in EXTRACT_FIELDS.keys():
        base[f] = best_span_text(r[f])
    flat_rows.append(base)

df_evidence_flat = pd.DataFrame(flat_rows)
df_evidence_flat.head(5)

Xuất kết quả (CSV) + bảng theo cluster

In [ ]:
OUT_DIR = "/content/output_sr"
os.makedirs(OUT_DIR, exist_ok=True)

df_cand.to_csv(f"{OUT_DIR}/candidates_with_clusters.csv", index=False)
df_inc.to_csv(f"{OUT_DIR}/included.csv", index=False)
df_evidence_flat.to_csv(f"{OUT_DIR}/evidence_table.csv", index=False)

# summary cluster
cluster_summary = (
    df_inc.groupby("cluster_id")
    .agg(n_papers=("paper_id","nunique"), top_title=("title", "first"))
    .reset_index()
    .sort_values("n_papers", ascending=False)
)
cluster_summary.to_csv(f"{OUT_DIR}/cluster_summary.csv", index=False)

print("Saved to:", OUT_DIR)
cluster_summary.head(10)